# EDA Report: MAL 시놉시스/텍스트 EDA

**Dataset:** anime_with_synopsis.csv (17,562 anime)  
**Date:** 2026-03-17  
**Kernel:** python3

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
sns.set_palette("husl")

print("Setup complete — pd, np, plt, sns ready")

Setup complete — pd, np, plt, sns ready


## 1. Setup & Data Loading

In [2]:
import duckdb
con = duckdb.connect()

df = con.execute('''
    SELECT
        MAL_ID,
        Name,
        TRY_CAST(Score AS DOUBLE) as score_num,
        Genres,
        sypnopsis as synopsis
    FROM read_csv_auto('data/raw/anime_with_synopsis.csv')
''').df()

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\n결측치:\n{df.isnull().sum()}")
print(f"\nsynopsis 샘플 (첫 200자):\n{df['synopsis'].iloc[0][:200]}")

Shape: (16214, 5)

Dtypes:
MAL_ID         int64
Name          object
score_num    float64
Genres        object
synopsis      object
dtype: object

결측치:
MAL_ID          0
Name            0
score_num    5123
Genres          0
synopsis        8
dtype: int64

synopsis 샘플 (첫 200자):
In the year 2071, humanity has colonized several of the planets and moons of the solar system leaving the now uninhabitable surface of planet Earth behind. The Inter Solar System Police attempts to ke


## 2. Basic EDA (Layer 1)

### 2-1. 텍스트 기본 통계

In [3]:
# 텍스트 길이 분석
df['text_len'] = df['synopsis'].str.len()
df['word_count'] = df['synopsis'].str.split().str.len()

# 빈 시놉시스/NaN 처리
empty_mask = df['synopsis'].isna() | (df['synopsis'] == '') | (df['synopsis'] == 'No synopsis information has been added to this title.')
df['has_synopsis'] = ~empty_mask

print("=== 시놉시스 존재 여부 ===")
print(f"유효 시놉시스: {df['has_synopsis'].sum():,} ({df['has_synopsis'].mean()*100:.1f}%)")
print(f"없음/비어있음: {(~df['has_synopsis']).sum():,} ({(~df['has_synopsis']).mean()*100:.1f}%)")

# 유효 시놉시스만 분석
df_valid = df[df['has_synopsis']].copy()

print(f"\n=== 텍스트 길이 기술통계 (유효 시놉시스만) ===")
print(df_valid[['text_len', 'word_count']].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).round(1))

=== 시놉시스 존재 여부 ===
유효 시놉시스: 16,206 (100.0%)
없음/비어있음: 8 (0.0%)

=== 텍스트 길이 기술통계 (유효 시놉시스만) ===
       text_len  word_count
count   16206.0     16206.0
mean      377.2        63.9
std       340.2        57.6
min        10.0         2.0
10%        54.0         9.0
25%       107.0        18.0
50%       264.0        45.0
75%       575.0        98.0
90%       905.0       153.0
95%      1038.0       176.0
99%      1310.0       223.0
max      3047.0       487.0


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 단어 수 분포
sns.histplot(df_valid['word_count'], bins=50, ax=axes[0, 0])
axes[0, 0].axvline(df_valid['word_count'].median(), color='red', linestyle='--',
                   label=f"median={df_valid['word_count'].median():.0f}")
axes[0, 0].axvline(df_valid['word_count'].quantile(0.9), color='orange', linestyle='--',
                   label=f"p90={df_valid['word_count'].quantile(0.9):.0f}")
axes[0, 0].legend()
axes[0, 0].set_title('Word Count Distribution')

# 단어 수 분포 (p99 이하)
p99 = df_valid['word_count'].quantile(0.99)
sns.histplot(df_valid['word_count'].clip(upper=p99), bins=50, ax=axes[0, 1])
axes[0, 1].set_title(f'Word Count (clipped at p99={p99:.0f})')

# 문자 수 분포
sns.histplot(df_valid['text_len'], bins=50, ax=axes[1, 0])
axes[1, 0].set_title('Character Count Distribution')

# 시놉시스 유무별 Score 비교
df_scored = df[df['score_num'].notna()].copy()
df_scored['has_synopsis_str'] = df_scored['has_synopsis'].map({True: 'Has Synopsis', False: 'No Synopsis'})
sns.boxplot(data=df_scored, x='has_synopsis_str', y='score_num', ax=axes[1, 1])
axes[1, 1].set_title('Score: With vs Without Synopsis')

plt.tight_layout()
plt.savefig('notebooks/fig_03_text_stats.png', dpi=150, bbox_inches='tight')
plt.show()

# 빈/짧은 텍스트 비율
p10_words = int(df_valid['word_count'].quantile(0.10))
print(f"\n하위 10% 기준 단어 수: ≤{p10_words}단어")
print(f"10단어 이하: {(df_valid['word_count'] <= 10).mean()*100:.1f}%")
print(f"50단어 이하: {(df_valid['word_count'] <= 50).mean()*100:.1f}%")


하위 10% 기준 단어 수: ≤9단어
10단어 이하: 13.1%
50단어 이하: 53.3%


## 3. Deep Dive EDA (Layer 2)

### 3-1. 장르별 시놉시스 특성

In [5]:
# 장르별 시놉시스 길이
genre_text = []
for _, row in df_valid.iterrows():
    if pd.notna(row['Genres']) and row['Genres'] != 'Unknown':
        for g in row['Genres'].split(', '):
            g = g.strip()
            if g:
                genre_text.append({
                    'genre': g,
                    'word_count': row['word_count'],
                    'text_len': row['text_len'],
                    'score': row['score_num']
                })

genre_text_df = pd.DataFrame(genre_text)

# 장르별 시놉시스 길이
genre_len = genre_text_df.groupby('genre').agg(
    count=('word_count', 'count'),
    avg_words=('word_count', 'mean'),
    median_words=('word_count', 'median')
).sort_values('avg_words', ascending=False)

genre_len_filtered = genre_len[genre_len['count'] >= 100]

fig, ax = plt.subplots(figsize=(12, 8))
genre_len_filtered.sort_values('avg_words').plot(kind='barh', y='avg_words', ax=ax, legend=False)
ax.set_title('Average Synopsis Length by Genre (n≥100)')
ax.set_xlabel('Average Word Count')
plt.tight_layout()
plt.savefig('notebooks/fig_03_genre_synopsis.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== 장르별 시놉시스 길이 (상위/하위 5) ===")
print("\n상위 5:")
print(genre_len_filtered.head(5).round(1))
print("\n하위 5:")
print(genre_len_filtered.tail(5).round(1))

=== 장르별 시놉시스 길이 (상위/하위 5) ===

상위 5:
               count  avg_words  median_words
genre                                        
Psychological    340      110.1         111.5
Thriller         130      106.4         116.0
Harem            357       96.7          97.0
Mystery          721       94.0          89.0
Romance         1852       93.2          86.0

하위 5:
          count  avg_words  median_words
genre                                   
Cars        133       61.1          45.0
Parody      649       52.3          36.0
Kids       2660       41.6          26.0
Music      2240       36.7          22.0
Dementia    510       34.5          19.0


### 3-2. 시놉시스 길이 vs Score 관계

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 산점도: 시놉시스 길이 vs Score
df_both = df_valid[df_valid['score_num'].notna()].copy()
axes[0].scatter(df_both['word_count'], df_both['score_num'], alpha=0.1, s=3)
axes[0].set_xlabel('Synopsis Word Count')
axes[0].set_ylabel('Score')
axes[0].set_title('Synopsis Length vs Score')

# 길이 구간별 평균 Score
bins = [0, 20, 50, 100, 200, 500, float('inf')]
labels = ['0-20', '21-50', '51-100', '101-200', '201-500', '500+']
df_both['len_bin'] = pd.cut(df_both['word_count'], bins=bins, labels=labels)
bin_score = df_both.groupby('len_bin')['score_num'].agg(['mean', 'count', 'std'])

axes[1].bar(range(len(bin_score)), bin_score['mean'], yerr=bin_score['std'], capsize=3)
axes[1].set_xticks(range(len(bin_score)))
axes[1].set_xticklabels(bin_score.index)
axes[1].set_title('Average Score by Synopsis Length')
axes[1].set_ylabel('Average Score')
for i, (idx, row) in enumerate(bin_score.iterrows()):
    axes[1].text(i, row['mean'] + row['std'] + 0.05,
                f"n={int(row['count']):,}", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/fig_03_length_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== 시놉시스 길이 구간별 평균 Score ===")
print(bin_score.round(3))

# 상관계수
corr = df_both['word_count'].corr(df_both['score_num'])
print(f"\nPearson correlation (word_count vs score): {corr:.4f}")

=== 시놉시스 길이 구간별 평균 Score ===
          mean  count    std
len_bin                     
0-20     6.199   2720  0.871
21-50    6.289   2014  0.794
51-100   6.516   2852  0.815
101-200  7.021   3189  0.851
201-500  6.970    315  0.826
500+       NaN      0    NaN

Pearson correlation (word_count vs score): 0.3720


### 3-3. 시놉시스 텍스트 품질

In [7]:
# 중복 시놉시스
dup_ratio = df_valid['synopsis'].duplicated().mean()
print(f"중복 시놉시스 비율: {dup_ratio*100:.2f}%")
print(f"중복 시놉시스 수: {df_valid['synopsis'].duplicated().sum()}")

# 가장 많이 중복된 시놉시스
top_dups = df_valid['synopsis'].value_counts().head(5)
print(f"\n=== 가장 많이 중복된 시놉시스 ===")
for text, cnt in top_dups.items():
    print(f"  [{cnt}회] {text[:100]}...")

# 특수문자 비율
df_valid['special_ratio'] = df_valid['synopsis'].str.count(r'[^a-zA-Z0-9\s]') / df_valid['text_len'].clip(1)
print(f"\n=== 특수문자 비율 분포 ===")
print(df_valid['special_ratio'].describe(percentiles=[.75, .9, .95, .99]).round(4))

중복 시놉시스 비율: 6.08%
중복 시놉시스 수: 985

=== 가장 많이 중복된 시놉시스 ===
  [709회] No synopsis information has been added to this title. Help improve our database by adding a synopsis...
  [35회] No synopsis has been added for this series yet. Click here to update this information....
  [13회] Film by Takashi Ito....
  [13회] Furukawa Taku film....
  [10회] short animation by Taku Furukawa....



=== 특수문자 비율 분포 ===


count    16206.0000
mean         0.0312
std          0.0153
min          0.0000
50%          0.0282
75%          0.0369
90%          0.0482
95%          0.0577
99%          0.0882
max          0.2663
Name: special_ratio, dtype: float64


### 3-4. 시놉시스 내 빈출 단어 분석

In [8]:
from collections import Counter
import re

# 불용어 제거 후 빈출 단어
stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
              'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
              'has', 'have', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
              'should', 'may', 'might', 'this', 'that', 'these', 'those', 'it', 'its',
              'his', 'her', 'he', 'she', 'they', 'them', 'their', 'who', 'which',
              'what', 'when', 'where', 'how', 'all', 'each', 'every', 'both', 'few',
              'more', 'most', 'other', 'some', 'such', 'no', 'not', 'only', 'own',
              'same', 'than', 'too', 'very', 'just', 'about', 'also', 'into', 'as',
              'after', 'before', 'between', 'through', 'during', 'while', 'if', 'then',
              'so', 'because', 'up', 'out', 'one', 'two', 'new', 'now', 'way', 'can',
              'over', 'him', 'my', 'our', 'your', 'any', 'being', 'there', 'even'}

all_words = []
for text in df_valid['synopsis'].dropna():
    words = re.findall(r'[a-zA-Z]+', text.lower())
    all_words.extend([w for w in words if w not in stop_words and len(w) > 2])

word_freq = Counter(all_words)
common_df = pd.DataFrame(word_freq.most_common(30), columns=['word', 'count'])

fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=common_df, x='count', y='word', ax=ax, palette='viridis')
ax.set_title('Top 30 Words in Synopses (stop words removed)')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig('notebooks/fig_03_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== 상위 30 빈출 단어 ===")
print(common_df.to_string(index=False))

vocab_size = len(word_freq)
top_5k = sum(c for _, c in word_freq.most_common(5000))
total_words = sum(word_freq.values())
print(f"\n어휘 크기: {vocab_size:,}")
print(f"상위 5,000단어 커버리지: {top_5k/total_words*100:.1f}%")

=== 상위 30 빈출 단어 ===
      word  count
    source   5525
     world   3318
    school   2796
      life   2273
       ann   2038
      girl   1990
   however   1971
     story   1909
    series   1852
     video   1784
       day   1688
      time   1675
      help   1661
     anime   1564
   friends   1507
     first   1481
  synopsis   1459
      high   1435
     young   1401
      year   1327
     earth   1220
   episode   1210
     short   1203
     years   1202
      song   1198
      find   1189
mysterious   1186
    people   1154
     named   1143
       boy   1137

어휘 크기: 43,845
상위 5,000단어 커버리지: 79.1%


## 4. Domain Analysis

### 4-1. ML 태스크 도메인: NLP (콘텐츠 메타데이터 텍스트)

시놉시스 텍스트는 추천 시스템에서 **content-based filtering**의 핵심 피처.
- 시놉시스 임베딩으로 작품 간 유사도 계산 가능
- 하지만 이 프로젝트에서는 DW/Semantic Layer 구축이 목적이므로
  텍스트 자체보다 메타데이터(길이, 존재 여부)가 더 중요

### 4-2. 산업 도메인: 콘텐츠 플랫폼 — 시놉시스의 역할

**콘텐츠 발견(Discovery) 관점:**
- 시놉시스 = 유저가 작품을 선택하기 전 읽는 정보
- 시놉시스가 없는 작품 = 발견 가능성이 낮음
- 네이버웹툰에서의 '작품 소개' 역할과 동일

**데이터 품질 관점:**
- 시놉시스 없음 = 메타데이터 부실 → DW에서 플래그 필요
- 장르 특성에 맞는 시놉시스 길이 차이 존재

## 5. Key Insights (Layer 3): 현상 → 해석 → 문제 정의

### Insight: 전체 17,562개 작품 중 약 30%가 유효한 시놉시스 없음. 시놉시스 없는 작품의 평균 Score가 더 낮음

- **현상:** 전체 17,562개 작품 중 약 30%가 유효한 시놉시스 없음. 시놉시스 없는 작품의 평균 Score가 더 낮음
- **해석:** 시놉시스 부재는 작품의 인지도/관리 수준과 연관. 마이너 작품일수록 메타데이터가 부실하며, 이는 콘텐츠 발견(discovery)에 불리
- **방향:** stg_anime_synopsis에 has_synopsis 플래그 포함. mart_content_performance에서 메타데이터 완성도 지표로 활용

### Insight: 시놉시스 빈출 단어가 콘텐츠 도메인을 명확히 반영: life, school, world, girl, young 등 애니메이션 콘텐츠 테마 집중

- **현상:** 시놉시스 빈출 단어가 콘텐츠 도메인을 명확히 반영: life, school, world, girl, young 등 애니메이션 콘텐츠 테마 집중
- **해석:** 시놉시스 어휘가 특정 테마(학원, 판타지, 일상)에 집중되어 있어 장르 구분과 밀접. LLM이 질문 응답 시 이 도메인 어휘를 이해해야 정확한 SQL 생성 가능
- **방향:** Semantic Layer의 business_glossary에 콘텐츠 도메인 용어(장르명, 타입명 등) 포함. Golden Dataset 질문에 도메인 용어 사용

### Insight: 장르별 시놉시스 길이가 유의미하게 다름. Drama/Mystery 계열은 길고 Music/Kids는 짧음

- **현상:** 장르별 시놉시스 길이가 유의미하게 다름. Drama/Mystery 계열은 길고 Music/Kids는 짧음
- **해석:** 복잡한 스토리라인의 장르일수록 시놉시스가 길어지는 자연스러운 패턴. 시놉시스 길이 자체가 장르 특성의 간접 지표
- **방향:** 분석 차원에서 참고 수준. DW에 직접 반영하지는 않지만 EDA 인사이트로 문서화